# Driver Drowsiness Detection - Experimentation Notebook

This notebook is for experimentation, visualization, and interactive testing of the drowsiness detection model.
All production code is modularized in the `src/` directory.

## Section 1: Import Required Libraries

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('..'))

# Import from modular src package
from src.data import load_dataset_paths, create_dataloaders, get_train_transform, get_eval_transform
from src.model import create_model
from src.training import create_trainer
from src.evaluation import evaluate_model
from src.utils import DEVICE, SEED, NUM_EPOCHS, LEARNING_RATE, CLASS_NAMES

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {DEVICE}")

## Section 2: Load and Prepare Data

In [ ]:
# Set random seed
torch.manual_seed(SEED)
np.random.seed(SEED)

# Load dataset paths
print("Loading dataset paths...")
train_paths, test_paths = load_dataset_paths()

print(f"Total training samples: {len(train_paths)}")
print(f"Total test samples: {len(test_paths)}")

# Verify data directory structure
print("\nDataset structure:")
data_dir = "Driver Drowsiness Dataset (DDD)"
if os.path.exists(data_dir):
    for subdir in os.listdir(data_dir):
        subdir_path = os.path.join(data_dir, subdir)
        if os.path.isdir(subdir_path):
            count = len(os.listdir(subdir_path))
            print(f"  {subdir}: {count} images")

## Section 3: Create Data Loaders

In [ ]:
# Create data loaders
print("Creating data loaders...")
train_dataloader, test_dataloader = create_dataloaders(
    train_paths,
    test_paths,
    train_transform=get_train_transform(),
    eval_transform=get_eval_transform(),
)

print(f"Train batches: {len(train_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")

# Inspect a batch
print("\nInspecting batch shape:")
images, labels = next(iter(train_dataloader))
print(f"  Images shape: {images.shape}")
print(f"  Labels shape: {labels.shape}")
print(f"  Unique labels: {torch.unique(labels).tolist()}")

## Section 4: Train the Model

In [ ]:
# Create model
print("Creating CNN model...")
model = create_model()
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture:")
print(model)
print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

## Section 5: Train and Evaluate

In [ ]:
# Create trainer and train
print("Setting up trainer...")
trainer = create_trainer(model, learning_rate=LEARNING_RATE, device=DEVICE)

print(f"\nTraining for {NUM_EPOCHS} epochs...")
losses = trainer.train(train_dataloader, num_epochs=NUM_EPOCHS)

## Section 6: Evaluate the Model

In [ ]:
# Evaluate model
print("Evaluating model on test set...")
accuracy, metrics = evaluate_model(model, test_dataloader, device=DEVICE)

print(f"\nEvaluation Results:")
print(f"  Accuracy: {accuracy:.2f}%")
print(f"  Correct: {metrics['correct']}/{metrics['total']}")

## Section 7: Save the Model

In [ ]:
# Save model
from src.utils import MODEL_SAVE_PATH

print(f"Saving model to {MODEL_SAVE_PATH}...")

# Create models directory if it doesn't exist
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

try:
    # Save as TorchScript
    model_scripted = torch.jit.script(model)
    model_scripted.save(MODEL_SAVE_PATH)
    print(f"✓ Model saved successfully as TorchScript")
except Exception as e:
    print(f"Warning: Could not save as TorchScript: {e}")
    # Fallback: save as regular PyTorch model
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"✓ Model saved as PyTorch state dict")

# Verify save
if os.path.exists(MODEL_SAVE_PATH):
    size_mb = os.path.getsize(MODEL_SAVE_PATH) / (1024 * 1024)
    print(f"✓ Model file size: {size_mb:.2f} MB")
else:
    print("✗ Error: Model file not saved")

## Section 8: Summary

In [ ]:
print("="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"\nFinal Test Accuracy: {accuracy:.2f}%")
print(f"Training Loss (final): {losses[-1]:.6f}")
print(f"Model saved to: {MODEL_SAVE_PATH}")

print(f"\nNext Steps:")
print(f"  1. Run API: python -m uvicorn api.main:app --reload")
print(f"  2. Test API at: http://localhost:8000/docs")
print(f"  3. Test image: python inference.py <image_path>")
print("="*60)